<a href="https://colab.research.google.com/github/ElofssonLab/kb8029-book/blob/main/notebooks/day08-lab.ipynb" style="display:inline-block;padding:10px 18px;background-color:#F9AB00;color:#000000;font-weight:bold;text-decoration:none;border-radius:6px;font-family:sans-serif;font-size:14px;">&#9654;&nbsp; Open in Google Colab</a>

# Day 8 lab: data, models and evaluation on MNIST {.unnumbered}

This lab is the hands-on half of Day 8. It uses **MNIST**, 70,000
handwritten digits (28×28 grey-scale pixels, labels 0-9), the classic
first dataset for neural networks, and then comes back to proteins for
the homology question at the end.

Every `todo("...")` call marks a piece of code for you to write: replace
the whole `todo(...)` call with your code. Work through the cells **in
order, top to bottom**. Until you fill it in, a cell stops with
`NotImplementedError: TODO in this cell: ...`, which tells you what is
missing. That is expected.

Day 8 does not teach *training* yet. The notebook gives you a ready-made
`train_model(...)` function; Day 9 opens that box (backpropagation, the
training loop, cross-entropy). Here you build the data sets and the
model, and you **evaluate** what comes out.

**Runtime:** a few minutes on Colab's free CPU. Keep `SEED = 0` so that
your numbers match the lab quiz (neural-network training still varies a
little between machines, so the quiz accepts a small range).

In [ ]:
import os, re, tempfile
import numpy as np
import matplotlib.pyplot as plt
import torch
import torch.nn as nn
import torchvision
from sklearn.metrics import confusion_matrix, roc_curve, roc_auc_score, matthews_corrcoef

SEED = 0
torch.manual_seed(SEED)
np.random.seed(SEED)
torch.set_num_threads(max(1, os.cpu_count() or 1))

# MNIST is downloaded to a temporary directory, not next to this notebook
DATA_DIR = os.path.join(tempfile.gettempdir(), "kb8029_mnist")

def todo(what):
    """Placeholder for code you write: replace the whole todo(...) call with your own code."""
    raise NotImplementedError(f"TODO in this cell: {what}")

## 1. Load MNIST and look at the classes

`torchvision` downloads the official split: 60,000 training images and
10,000 test images. The images come as integers 0-255.

In [ ]:
mnist_train = torchvision.datasets.MNIST(DATA_DIR, train=True, download=True)
mnist_test = torchvision.datasets.MNIST(DATA_DIR, train=False, download=True)

X_all = mnist_train.data.numpy().astype(np.float32)     # (60000, 28, 28)
y_all = mnist_train.targets.numpy()                     # (60000,)
X_test = mnist_test.data.numpy().astype(np.float32)     # (10000, 28, 28)
y_test = mnist_test.targets.numpy()
print("training images:", X_all.shape, " test images:", X_test.shape)
print("pixel value range:", X_all.min(), "-", X_all.max())

fig, axes = plt.subplots(1, 10, figsize=(12, 1.6))
for digit, ax in enumerate(axes):
    ax.imshow(X_all[np.where(y_all == digit)[0][0]], cmap="gray_r")
    ax.set_title(str(digit)); ax.axis("off")
plt.show()

In [ ]:
class_counts = todo('count how many training images there are of each digit (hint: np.bincount)')
for digit, n in enumerate(class_counts):
    print(f"digit {digit}: {n} images")
most_common = todo('the digit with the most images')
print("most abundant digit:", most_common, " least abundant:", int(np.argmin(class_counts)))

## 2. Training, validation and test sets

We don't need all 60,000 images to learn something. `train_size_per_class`
controls how many images *per digit* go into the training set; another
100 per digit (never used for training) form the **validation set**.
The official 10,000-image **test set** stays untouched until Section 8.

In [ ]:
train_size_per_class = 500
val_size_per_class = 100

def make_split(train_per_class, val_per_class=val_size_per_class, seed=SEED):
    rng = np.random.RandomState(seed)
    train_idx, val_idx = [], []
    for digit in range(10):
        idx = rng.permutation(np.where(y_all == digit)[0])
        val_idx.append(idx[:val_per_class])                                  # validation comes first ...
        train_idx.append(idx[val_per_class:val_per_class + train_per_class]) # ... so it never changes with train size
    return np.concatenate(train_idx), np.concatenate(val_idx)

train_idx, val_idx = make_split(train_size_per_class)
print("training images:", len(train_idx), " validation images:", len(val_idx), " test images:", len(y_test))

## 3. Normalization, without leaking the test set

A common strategy is to **standardize**: subtract the mean and divide by
the standard deviation, so that inputs are centred on zero with unit
spread. Here we use one mean and one standard deviation for all pixels.
Day 8's rule: preprocessing statistics come from the **training set
only**, and are then applied unchanged to validation and test data.

In [ ]:
X_train_raw = X_all[train_idx] / 255.0
X_val_raw = X_all[val_idx] / 255.0
X_test_raw = X_test / 255.0

pixel_mean = todo('the mean pixel value of the TRAINING images only')
pixel_std = todo('the standard deviation of the TRAINING images only')
print(f"training-set pixel mean: {pixel_mean:.4f}   std: {pixel_std:.4f}")

standardize = lambda X: (X - pixel_mean) / pixel_std
X_train, X_val, X_testn = standardize(X_train_raw), standardize(X_val_raw), standardize(X_test_raw)
y_train, y_val = y_all[train_idx], y_all[val_idx]
print(f"after standardizing: train mean {X_train.mean():+.4f}, std {X_train.std():.4f}")
print(f"                     val   mean {X_val.mean():+.4f}, std {X_val.std():.4f}   (close to, not exactly, 0 and 1)")

## 4. The model, as an `nn.Module`

Write a class with these layers, in order:

1. `nn.Flatten()`: a 28×28 image becomes a vector of 784 numbers;
2. `nn.Linear(784, 10)` followed by `nn.Tanh()`: a hidden layer of 10 units;
3. `nn.Linear(10, 10)`: 10 outputs, one **logit** per digit.

There is no softmax layer: as on the Day 8 page, the loss used inside
`train_model` applies it internally.

In [ ]:
class DigitClassifier(nn.Module):
    def __init__(self):
        super().__init__()
        self.net = todo('nn.Sequential of Flatten, Linear(784, 10), Tanh, Linear(10, 10)')

    def forward(self, x):
        return self.net(x)

model = DigitClassifier()
print(model)
n_params = todo('total number of trainable parameters (sum of p.numel())')
print("trainable parameters:", n_params)

## 5. Training (a black box until Day 9)

`train_model` runs **stochastic gradient descent** in mini-batches of 128
images and records training and validation accuracy after every epoch.
You don't need to change it.

In [ ]:
def train_model(model, X_train, y_train, X_val, y_val, epochs=30, lr=0.1, batch_size=128, seed=SEED):
    '''Train `model` with mini-batch SGD and cross-entropy loss (Day 9 explains
    every line). Returns a dict of per-epoch training/validation accuracy.'''
    torch.manual_seed(seed)
    Xt, yt = torch.tensor(X_train), torch.tensor(y_train)
    Xv, yv = torch.tensor(X_val), torch.tensor(y_val)
    optimizer = torch.optim.SGD(model.parameters(), lr=lr)
    loss_fn = nn.CrossEntropyLoss()
    history = {"train_acc": [], "val_acc": []}
    for epoch in range(epochs):
        model.train()
        for batch in torch.randperm(len(Xt)).split(batch_size):
            optimizer.zero_grad()
            loss_fn(model(Xt[batch]), yt[batch]).backward()
            optimizer.step()
        model.eval()
        with torch.no_grad():
            history["train_acc"].append((model(Xt).argmax(1) == yt).float().mean().item())
            history["val_acc"].append((model(Xv).argmax(1) == yv).float().mean().item())
    return history

torch.manual_seed(SEED)
model = DigitClassifier()
history = train_model(model, X_train, y_train, X_val, y_val)
print(f"after {len(history['train_acc'])} epochs: training accuracy {history['train_acc'][-1]:.3f}, "
      f"validation accuracy {history['val_acc'][-1]:.3f}")

plt.plot(history["train_acc"], label="training")
plt.plot(history["val_acc"], label="validation")
plt.xlabel("epoch"); plt.ylabel("accuracy"); plt.legend(); plt.show()

## 6. From outputs to classes, and the confusion matrix

The model outputs 10 logits per image. `torch.softmax` turns them into
10 probabilities that sum to 1; the predicted class is the one with the
highest probability.

In [ ]:
model.eval()
with torch.no_grad():
    logits = model(torch.tensor(X_val))
probs = todo('softmax over the 10 outputs (dim=1), as a NumPy array')
predicted = todo('the predicted digit for every validation image')
print("first image's probabilities:", np.round(probs[0], 3), " sum =", probs[0].sum().round(4))
print("first 10 predictions:", predicted[:10], "  true labels:", y_val[:10])

val_accuracy = todo('fraction of validation images predicted correctly')
print(f"validation accuracy: {val_accuracy:.3f}")

cm = confusion_matrix(y_val, predicted)
print("confusion matrix (rows = true digit, columns = predicted digit):")
print(cm)

### Digit 2 against the rest

Treat "2" as the positive class and every other digit as negative:

- predicted 2, truly 2: **true positive (TP)**
- predicted 2, truly not 2: **false positive (FP)**
- predicted not 2, truly 2: **false negative (FN)**
- predicted not 2, truly not 2: **true negative (TN)**

In [ ]:
is_2, pred_2 = (y_val == 2), (predicted == 2)
TP = todo('count of true positives')
FP = todo('count of false positives')
FN = todo('count of false negatives')
TN = todo('count of true negatives')
print(f"TP={TP}  FP={FP}  FN={FN}  TN={TN}   (total {TP+FP+FN+TN})")

precision = todo('precision for digit 2')
recall = todo('recall for digit 2')
f1 = todo('F1 for digit 2')
mcc = matthews_corrcoef(is_2, pred_2)
print(f"digit 2: precision {precision:.3f}  recall {recall:.3f}  F1 {f1:.3f}  MCC {mcc:.3f}")
print(f"accuracy of the 2-vs-rest call: {(TP + TN) / len(y_val):.3f}")

## 7. ROC curve and AUROC for digit 2

The ROC curve needs a *score*, not a yes/no call: use the model's
probability for class 2.

In [ ]:
score_2 = todo('the predicted probability of class 2 for every validation image')
fpr, tpr, _ = roc_curve(is_2, score_2)
auroc_2 = roc_auc_score(is_2, score_2)
plt.plot(fpr, tpr, label=f"digit 2 vs rest (AUROC {auroc_2:.3f})")
plt.plot([0, 1], [0, 1], "--", color="grey", label="random")
plt.xlabel("false-positive rate"); plt.ylabel("true-positive rate"); plt.legend(); plt.show()
print(f"AUROC, digit 2 vs rest: {auroc_2:.4f}")

## 8. How much data does the model need? Overfitting

Retrain the same model with different training-set sizes. The validation
set stays the same 1,000 images every time. Watch the gap between
training and validation accuracy.

In [ ]:
sizes = [10, 50, 200, 500]
results = {}
for n in sizes:
    tr_idx, _ = make_split(n)
    Xtr = standardize(X_all[tr_idx] / 255.0)
    torch.manual_seed(SEED)
    m = DigitClassifier()
    h = train_model(m, Xtr, y_all[tr_idx], X_val, y_val, epochs=30)
    results[n] = (h["train_acc"][-1], h["val_acc"][-1])
    print(f"{n:4d} per class ({10*n:5d} images): training acc {results[n][0]:.3f}   validation acc {results[n][1]:.3f}"
          f"   gap {results[n][0] - results[n][1]:+.3f}")

plt.plot(sizes, [results[n][0] for n in sizes], "o-", label="training")
plt.plot(sizes, [results[n][1] for n in sizes], "o-", label="validation")
plt.xscale("log"); plt.xlabel("training images per digit"); plt.ylabel("accuracy after 30 epochs")
plt.legend(); plt.show()

### The test set, used once

Validation has guided our choices (e.g. the training-set size). Now,
**once**, measure the 500-per-digit model from Section 5 on the official
10,000 test images:

In [ ]:
with torch.no_grad():
    test_pred = model(torch.tensor(X_testn)).argmax(1).numpy()
test_accuracy = todo('accuracy on the 10,000 test images')
print(f"test accuracy: {test_accuracy:.3f}   (validation accuracy was {val_accuracy:.3f})")

## 9. Back to proteins: how leaky is a random split?

Day 8's localization dataset: 700 human proteins in 5 compartments,
clustered with MMseqs2 at 30% identity (precomputed, so no MMseqs2 is
needed). A **random** split ignores the clusters. How many validation
proteins end up with a homolog, i.e. a member of the same cluster, in
the training set? This needs only the accessions and labels (fetched
live from UniProt, ~10 s), no model.

In [ ]:
import requests
from sklearn.model_selection import train_test_split

UNIPROT_CLASSES = {"Cytoplasm": "SL-0086", "Nucleus": "SL-0191", "Mitochondrion": "SL-0173",
                   "Secreted": "SL-0243", "Cell membrane": "SL-0039"}
CLASS_NAMES = list(UNIPROT_CLASSES)

def fetch_uniprot(sl_code, size=500):
    query = f"organism_id:9606 AND reviewed:true AND cc_scl_term:{sl_code} AND length:[50 TO 500]"
    r = requests.get("https://rest.uniprot.org/uniprotkb/search",
                     params={"query": query, "fields": "accession,sequence,cc_subcellular_location",
                             "format": "tsv", "size": size}, timeout=60)
    r.raise_for_status()
    return [tuple(line.split("\t")) for line in r.text.strip().split("\n")[1:] if len(line.split("\t")) == 3]

def is_unambiguous_single_location(location_text, target):
    location_text = re.sub(r"Note=.*", "", location_text)
    location_text = re.sub(r"\{[^}]*\}", "", location_text)
    if "Isoform" in location_text:
        return False
    terms = set()
    for statement in [s.strip() for s in location_text.split(".") if s.strip()]:
        body = statement.split(":", 1)[-1] if ":" in statement else statement
        top_term = re.split(r"[,;]", body)[0].strip()
        if top_term:
            terms.add(top_term)
    return len(terms) == 1 and next(iter(terms)) == target

entries = {c: [(a, s) for a, s, loc in fetch_uniprot(code) if is_unambiguous_single_location(loc, c)]
           for c, code in UNIPROT_CLASSES.items()}
n_per_class = min(len(v) for v in entries.values())
rng = np.random.RandomState(0)
accs, labels = [], []
for k, c in enumerate(CLASS_NAMES):
    for i in rng.choice(len(entries[c]), n_per_class, replace=False):
        accs.append(entries[c][i][0]); labels.append(k)
labels = np.array(labels)

TSV_URL = "https://raw.githubusercontent.com/ElofssonLab/kb8029-book/main/notebooks/data/day08-subcell-mmseqs-30.tsv"
TSV_LOCAL = "data/day08-subcell-mmseqs-30.tsv"
text = open(TSV_LOCAL).read() if os.path.exists(TSV_LOCAL) else requests.get(TSV_URL, timeout=30).text
member_to_cluster = {m: rep for rep, m in (line.split("\t") for line in text.strip().split("\n")[1:])}
clusters = np.array([member_to_cluster.get(a, a) for a in accs])
print(len(accs), "proteins in", len(set(clusters)), "clusters")

In [ ]:
idx = np.arange(len(labels))
tr, tmp = train_test_split(idx, test_size=0.30, stratify=labels, random_state=0)
va, te = train_test_split(tmp, test_size=0.5, stratify=labels[tmp], random_state=0)

train_clusters = set(clusters[tr])
has_homolog = todo('for each validation protein, is its cluster also in the training set?')
leak_fraction = todo('the fraction of validation proteins with a homolog in training')
print(f"random split (random_state=0): {has_homolog.sum()} of {len(va)} validation proteins "
      f"({leak_fraction:.1%}) have a >=30%-identity homolog in the training set")

**Last step:** save your notebook with all outputs (File → Download →
.ipynb) and upload it with the lab quiz on Canvas. If one of your numbers
falls outside the quiz's accepted range, the notebook lets us see whether
that is ordinary run-to-run variation in training.